# 04 逻辑回归 Logistic Regression

依赖安装说明：`pip install numpy matplotlib scikit-learn`

逻辑回归用于分类，尤其是二分类。虽然名字里有“回归”，但它输出的是类别概率。


## 1. 数学逻辑

先做一个线性打分：

$$z = w^Tx + b$$

再用 sigmoid 把打分压到 0 到 1：

$$p(y=1|x)=\sigma(z)=\frac{1}{1+e^{-z}}$$

二分类交叉熵损失是：

$$L = -\frac{1}{n}\sum_i[y_i\log(p_i)+(1-y_i)\log(1-p_i)]$$

如果 `p >= 0.5`，通常预测为 1，否则预测为 0。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
X, y = make_classification(n_samples=250, n_features=2, n_redundant=0, n_informative=2,
                           n_clusters_per_class=1, class_sep=1.4, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

plt.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolor='k')
plt.title('二分类数据')
plt.show()


In [ ]:
# 从零实现：sigmoid + BCE + 梯度下降

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

w = np.zeros(X_train_s.shape[1])
b = 0.0
lr = 0.1
loss_history = []

for step in range(400):
    z = X_train_s @ w + b
    p = sigmoid(z)
    loss = -np.mean(y_train * np.log(p + 1e-12) + (1 - y_train) * np.log(1 - p + 1e-12))
    loss_history.append(loss)

    grad_z = p - y_train
    grad_w = X_train_s.T @ grad_z / len(X_train_s)
    grad_b = np.mean(grad_z)
    w -= lr * grad_w
    b -= lr * grad_b

print('从零训练 w:', np.round(w, 3))
print('从零训练 b:', round(b, 3))
print('最终 loss:', round(loss_history[-1], 3))

plt.plot(loss_history)
plt.title('逻辑回归训练 loss')
plt.xlabel('step')
plt.ylabel('BCE')
plt.show()


In [ ]:
model = LogisticRegression()
model.fit(X_train_s, y_train)
pred = model.predict(X_test_s)
prob = model.predict_proba(X_test_s)[:, 1]

print('Accuracy:', round(accuracy_score(y_test, pred), 3))
print('Precision:', round(precision_score(y_test, pred), 3))
print('Recall:', round(recall_score(y_test, pred), 3))
print('Confusion matrix:')
print(confusion_matrix(y_test, pred))

xx, yy = np.meshgrid(np.linspace(X_train_s[:, 0].min()-1, X_train_s[:, 0].max()+1, 160),
                     np.linspace(X_train_s[:, 1].min()-1, X_train_s[:, 1].max()+1, 160))
grid = np.c_[xx.ravel(), yy.ravel()]
zz = model.predict_proba(grid)[:, 1].reshape(xx.shape)
plt.contourf(xx, yy, zz, levels=20, cmap='coolwarm', alpha=0.35)
plt.scatter(X_train_s[:, 0], X_train_s[:, 1], c=y_train, cmap='coolwarm', edgecolor='k')
plt.title('逻辑回归的概率决策面')
plt.show()


## 2. 常见误区

- 逻辑回归的决策边界默认是线性的；不是所有分类问题都适合。
- 类别严重不平衡时，accuracy 可能误导，要看 precision、recall、F1。
- 输出概率需要校准时，要额外做 calibration，不要默认它总是可靠概率。

## 3. 小实验

- 改 `class_sep`，观察分类难度变化。
- 把阈值从 `0.5` 改成 `0.3`，观察 recall 和 precision。
- 加多项式特征，让线性模型获得非线性边界。
